# CardGenius: AI That Knows Your Next Swipe

## 🧠 Problem Statement

In today’s fast-paced, cashless world, most of us carry multiple credit cards — each with its own reward categories, point systems, and rotating promos. But keeping track of which card gives the best rewards for a specific purchase can be a hassle — and most of us don’t have time to micromanage every transaction.

At the same time, decision fatigue is real. We wanted to tackle this **everyday micro-decision** with something smarter — and use the power of **Generative AI** to do it.

## 💡 Our Solution

**CardGenius** is a GenAI-driven recommendation engine that tells you **which credit card to use** for a given purchase — based on your personal cards and real-time reward info.

We built this to demonstrate the power of **Generative AI + RAG (Retrieval-Augmented Generation)** in solving simple but annoying real-world problems.

### 🔧 How It Works

1. **Store Card Info**  
   We use **[Chroma](https://www.trychroma.com/)** ,an open-source AI application database, to store just the names of the credit cards you own.  
   ➤ _No sensitive data, no account numbers — just the card names_ (e.g., “Chase Sapphire Preferred”, “Amex Gold”).

2. **Ask a Simple Question**  
   Ask things like:  
   - *“Which card should I use for gas?”*  
   - *“Best card for grocery shopping today?”*

3. **AI + Web + Grounding = Smart & Reliable Answer**  
   - The system pulls your card list from Chroma DB  
   - It performs a **Google search** to get the latest reward info  
   - GenAI processes the info, grounds it for accuracy, and gives you a **justified recommendation**

### ✅ Why It Works

- **No Risk, No Data Drama**  
  ➤ _Lightweight and private — no spending data, no logins, no dashboards._

- **Real-Time, Not Static**  
  ➤ _Credit card offers and promos change constantly. We use live search to stay up-to-date._

> ⚠️ _Note_: This MVP doesn't track spending caps or point limits (e.g., 5x on groceries up to $1,500/quarter). A secure transaction system would be needed for that. We’ve focused on solving the **general-use case** efficiently.

## Authors:
- Kishan Patel: [Kaggle-Kishan Patel](https://www.kaggle.com/kishanpatelai) | [LinkedIn](https://www.linkedin.com/in/kishan-patel-dev/)

## 🔑 Keywords

- **Multi-turn Chat**: Enables an interactive loop for ongoing user queries.
- **Google Search Grounding**: Uses real-time web search to ensure up-to-date and accurate information.
- **GenAI**: Leverages Generative AI (Gemini) to interpret and respond to user queries.
- **Function Calling**: Invokes specific functions like `get_best_card_recommendation` based on user input.
- **Retrieval-Augmented Generation (RAG)**: Combines retrieved data (search results) with LLM-generated responses.
- **Chroma**: Lightweight vector database used to store user credit card names.
- **LLM (Large Language Model)**: Powers the reasoning of the chatbot and recommendation engine.
- **Controlled Generation**: Prompt engineering + grounding ensure focused, reliable output from the LLM.

## Install the SDK
Installing ChromaDB and the Gemini API Python SDK.

In [1]:
# Uninstall packages from Kaggle base image that are not needed.
!pip uninstall -qy jupyterlab jupyterlab-lsp
# Remove unused conflicting packages
!pip uninstall -qqy jupyterlab kfp
# Install the google-genai SDK for this codelab.
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"

Import the SDK and some helpers for rendering the output.

In [2]:
from google import genai
from google.genai import types

from IPython.display import Markdown, HTML, display

genai.__version__

'1.7.0'

Set up a retry helper. This allows you to "Run all" without worrying about per-minute quota.

In [3]:
# Define a retry policy. The model might make multiple consecutive calls automatically
# for a complex query, this ensures the client retries if it hits quota limits.
from google.api_core import retry

is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

if not hasattr(genai.models.Models.generate_content, '__wrapped__'):
  genai.models.Models.generate_content = retry.Retry(
      predicate=is_retriable)(genai.models.Models.generate_content)

## Set up the API key
We are using the free API key from [AI Studio](https://aistudio.google.com/app/apikey). You can find [detailed instructions in the docs](https://ai.google.dev/gemini-api/docs/api-key) to get your own API key.

In [4]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

client = genai.Client(api_key=GOOGLE_API_KEY)

## Chroma DB
First we import and create a chroma client. Then we create our collection.

Collections are where we'll store our embeddings, documents, and any additional metadata. Collections index ours embeddings and documents, and enable efficient retrieval and filtering. ref: https://docs.trychroma.com/docs/overview/getting-started

In [5]:
# import and create a Chroma Client
import chromadb
chroma_client = chromadb.Client()

#create a collection
collection = chroma_client.create_collection(name="my_credit_card_collection")

Since in this scenario Chroma is passed a list of documents, it will automatically tokenize and embed them with the collection's embedding function 
(the default will be used if none was supplied at collection creation). Ref: https://docs.trychroma.com/docs/collections/add-data

By default, Chroma uses the Sentence Transformers all-MiniLM-L6-v2 model to create embeddings. This embedding model can create sentence and document embeddings that can be used for a wide variety of tasks. This embedding function runs locally on your machine, and may require you download the model files (this will happen automatically). Ref: https://docs.trychroma.com/docs/embeddings/embedding-functions

In [6]:
collection.upsert(
    documents=[
        "Chase Freedom Flex", 
        "U.S. Bank Altitude Go Visa Signature Card", 
        "Chase Ink Business Unlimited",
        "Chase Ink Business Cash",
        "Chase Ink Business Preferred",
        "Bilt Mastercard"
    ],
     #there's not an infinite number of credit cards a reasonable person can own so ids such as this should be manageable. We can also generate guids for this. But keeping it simple now
     ids=["id1", "id2", "id3", "id4","id5","id6"]
)

In [7]:
collection.count()
# You can peek at the data too.
#collection.peek(1)

6

In [8]:
def get_credit_card_names():
    results = collection.get()
    return results['documents']
    
# Here we can see all the credit cards we have so far in our db
get_credit_card_names()

['Chase Freedom Flex',
 'U.S. Bank Altitude Go Visa Signature Card',
 'Chase Ink Business Unlimited',
 'Chase Ink Business Cash',
 'Chase Ink Business Preferred',
 'Bilt Mastercard']

## Use search grounding
### Model support

Search grounding is available in a limited set of models. Find a model that supports it on the [models page](https://ai.google.dev/gemini-api/docs/models). In this project, we'll use **gemini-2.0-flash.**

### Make a request
To enable search grounding, we specify it as a tool: google_search. Like other tools, this is supplied as a parameter in GenerateContentConfig, and can be passed to generate_content calls as well as chats.create (for all chat turns) or chat.send_message (for specific turns).

### Grounding response
When search grounding is used, the model returns extra metadata that includes links to search suggestions, supporting documents and information on how the supporting documents were used. Each "grounding chunk" represents information retrieved from Google Search that was used in the grounded generation request. 

In [9]:
# Trying the query with search grounding enabled.
config_with_search = types.GenerateContentConfig(
    tools=[types.Tool(google_search=types.GoogleSearch())],
    temperature=0.0,
)

def query_with_grounding(query, metadata=False):
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=query,
        config=config_with_search,
    )
    if metadata: #this returns metadata response for us to see the grounding uris
        return response.candidates[0]
    else:
        return response.candidates[0].content.parts[0].text

The `get_best_card_recommendation` function helps determine the **best credit card to use** for a specific purchase scenario using real-time search and GenAI. Here's a breakdown of what happens:

1. **Creates a Google Search Query**  
   - Combines the user’s input query (e.g., `"grocery shopping"`),  
   - A list of credit cards the user owns,  
   - And the current month and year to ensure **data recency**.

2. **Performs a Grounded Web Search**  
   - Uses the generated query to search Google via `query_with_grounding`,  
   - This helps fetch **up-to-date** reward info for those specific cards.

3. **Generates a GenAI Prompt**  
   - Constructs a detailed prompt for Gemini that includes:
     - The search results,
     - The user's credit card list,
     - A request to **analyze** and **recommend** the best card to use today, with reasoning.

4. **Returns an AI-Powered Recommendation**  
   - Sends the prompt to the `gemini-2.0-flash` model,  
   - The model responds with a **card recommendation** and a brief **justification**.

```python
# Function call example:
get_best_card_recommendation("grocery shopping", ["Chase Freedom", "Amex Gold", "Citi Custom Cash"])


In [12]:
from datetime import date
def get_best_card_recommendation(query, credit_cards, verbose):
    current_date_str = date.today().strftime("%B %Y") # to ensure data recency
    search_query = f"best credit card rewards for {query} {', '.join(credit_cards)} {current_date_str}"
    if verbose:
        print("Google Search query:", search_query)
    search_results = query_with_grounding(search_query)
    if verbose:
        print("\nSearch results:", search_results)

    analysis_prompt = f"""Based on the following search results about credit card rewards for {query}:
'{search_results}'
and the following list of my credit cards:
'{credit_cards}',
which credit card should I use for {query} today to maximize value? Select the best one to optimize reward and explain your reasoning and be concise. You must suggest one."""

    recommendation_response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=[analysis_prompt]
    )
    return recommendation_response.text

## The Multi-Turn Chat Interface

The function 'chat_interface()' allows the user to input multiple queries in a loop. After each question, the chatbot processes the input, returns a recommendation, and then waits for the next input.

The conversation only ends when the user types 'exit'

In [14]:
def chat_interface(verbose=False):
    print("Welcome to Your Credit Card Recommendation Chatbot!")
    credit_cards = get_credit_card_names()
    if not credit_cards:
        print("No credit card names found in the database. Please run the initial setup.")
        return

    while True:
        #print("#"*130)
        user_input = input("What would you like to know? (e.g., 'Which card for groceries?', 'Best card for travel?', 'exit'): ")
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break
        elif user_input:
            recommendation = get_best_card_recommendation(user_input, credit_cards, verbose)
            print("*"*80)
            print("\nRecommendation:")
            display(Markdown(recommendation))
        else:
            print("Please enter a valid query.")
if __name__ == "__main__":
    chat_interface()

Welcome to Your Credit Card Recommendation Chatbot!


What would you like to know? (e.g., 'Which card for groceries?', 'Best card for travel?', 'exit'):  best credit card for flight ticket purchase?


********************************************************************************

Recommendation:


Based on the search results and your existing cards, the **Chase Ink Business Preferred** is likely the best option for flight ticket purchases. Here's why:

*   **High Reward Potential:** The search results indicate it's "best for business travelers" and good for "maximizing business purchases," implying a high rewards rate on travel, even though the exact rate isn't explicitly stated in this snippet.
*   **Travel Perks:** Business travel cards generally offer travel-related perks, making it suitable for flight purchases.
*   **Your Existing Card:** It's already in your wallet, eliminating the need to apply for a new card.

While other cards like the Chase Sapphire Preferred or Capital One Venture cards offer excellent rewards, you don't currently have them. Of the cards you listed, the Chase Ink Business Preferred is most suited for flight purchases.


What would you like to know? (e.g., 'Which card for groceries?', 'Best card for travel?', 'exit'):  exit


Goodbye!


## Verbose Reasoning using Grounding
Let's take a look at how the model reasons with more explicit print statements

In [15]:
def chat_interface(verbose=False):
    print("Welcome to Your Credit Card Recommendation Chatbot!")
    credit_cards = get_credit_card_names()
    if not credit_cards:
        print("No credit card names found in the database. Please run the initial setup.")
        return

    while True:
        #print("#"*130)
        user_input = input("What would you like to know? (e.g., 'Which card for groceries?', 'Best card for travel?', 'exit'): ")
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break
        elif user_input:
            recommendation = get_best_card_recommendation(user_input, credit_cards, verbose)
            print("*"*80)
            print("\nRecommendation:")
            display(Markdown(recommendation))
        else:
            print("Please enter a valid query.")
if __name__ == "__main__":
    chat_interface(True)

Welcome to Your Credit Card Recommendation Chatbot!


What would you like to know? (e.g., 'Which card for groceries?', 'Best card for travel?', 'exit'):  best credit card for rent?


Google Search query: best credit card rewards for best credit card for rent? Chase Freedom Flex, U.S. Bank Altitude Go Visa Signature Card, Chase Ink Business Unlimited, Chase Ink Business Cash, Chase Ink Business Preferred, Bilt Mastercard April 2025

Search results: Okay, I'll look into the best credit card rewards for paying rent as of April 2025, considering the cards you listed: Chase Freedom Flex, U.S. Bank Altitude Go Visa Signature Card, Chase Ink Business Unlimited, Chase Ink Business Cash, Chase Ink Business Preferred, and Bilt Mastercard.


********************************************************************************

Recommendation:


Bilt Mastercard.

Reasoning: The Bilt Mastercard is specifically designed for paying rent and offers rewards for doing so without transaction fees, making it the optimal choice among your listed cards. The others don't typically offer rewards on rent or may incur fees that negate the value of any rewards earned.


What would you like to know? (e.g., 'Which card for groceries?', 'Best card for travel?', 'exit'):  exit


Goodbye!


# Response Metadata
When search grounding is used, the model returns extra metadata that includes links to search suggestions, supporting documents and information on how the supporting documents were used.

Each "grounding chunk" represents information retrieved from Google Search that was used in the grounded generation request. Following the URI will take you to the source. 

Let's inspect some of those here for our query. This shows us that it is not hallucinating.

In [16]:
credit_cards = get_credit_card_names()
user_input = 'best credit card for rent'
verbose = False
#recommendation = get_best_card_recommendation(user_input, credit_cards, verbose)
current_date_str = date.today().strftime("%B %Y") # to ensure data recency
search_query = f"best credit card rewards for {user_input} {', '.join(credit_cards)} {current_date_str}"
search_results = query_with_grounding(search_query, True)

In [30]:
#search_results.grounding_metadata.search_entry_point.rendered_content
#HTML(search_results.grounding_metadata.search_entry_point.rendered_content)

The grounding_supports in the metadata provide a way for us to correlate the grounding chunks used to the generated output text.

In [32]:
from pprint import pprint

supports = search_results.grounding_metadata.grounding_supports
#for support in supports: #uncomment this to see the long list in print statement
    #pprint(support.to_json_dict())

In [28]:
while not search_results.grounding_metadata.grounding_supports or not search_results.grounding_metadata.grounding_chunks:
    # If incomplete grounding data was returned, retry.
    search_results = query_with_grounding()

chunks = search_results.grounding_metadata.grounding_chunks
#for chunk in chunks:
    #print(f'{chunk.web.title}: {chunk.web.uri}')

Below we print the supported text as well as citations to ensure that the model is not halucinating. 

In [29]:
import io

markdown_buffer = io.StringIO()

# Print the text with footnote markers.
markdown_buffer.write("Supported text:\n\n")
for support in supports:
    markdown_buffer.write(" * ")
    markdown_buffer.write(
        search_results.content.parts[0].text[support.segment.start_index : support.segment.end_index]
    )

    for i in support.grounding_chunk_indices:
        chunk = chunks[i].web
        markdown_buffer.write(f"<sup>[{i+1}]</sup>")

    markdown_buffer.write("\n\n")


# And print the footnotes.
markdown_buffer.write("Citations:\n\n")
for i, chunk in enumerate(chunks, start=1):
    markdown_buffer.write(f"{i}. [{chunk.web.title}]({chunk.web.uri})\n")


Markdown(markdown_buffer.getvalue())

Supported text:

 * *   **Rent Day Perks:** On the first of each month ("Rent Day"), Bilt offers special promotions:<sup>[1]</sup><sup>[2]</sup><sup>[3]</sup>

 * *   Double points on non-rent purchases (6x on dining, 4x on travel, 2x on other purchases, up to a 1,000 point bonus).<sup>[4]</sup><sup>[3]</sup><sup>[2]</sup>

 * *   For April 2025, there's a transfer bonus to Avios currencies (British Airways, Iberia, and Aer Lingus) of 50-100%, depending on your Bilt status.<sup>[3]</sup><sup>[5]</sup>

 * *   3 points per $1 on dining.<sup>[6]</sup><sup>[5]</sup><sup>[7]</sup>

 * *   2 points per $1 on travel.<sup>[5]</sup>

 * *   1 point per $1 on all other purchases.<sup>[8]</sup><sup>[9]</sup><sup>[10]</sup><sup>[11]</sup><sup>[12]</sup>

 * *   **Other Perks:** Bilt also offers opportunities to redeem points for Lyft rides with bonus value and access to exclusive dining and comedy experiences.<sup>[2]</sup><sup>[5]</sup><sup>[4]</sup>

 * *   **Requirement:** You need to use the card five times each statement period to earn points.<sup>[4]</sup><sup>[3]</sup>

 * Chase Freedom Flex**<sup>[13]</sup><sup>[12]</sup><sup>[14]</sup>

 * *   **Quarterly Categories:** Earns 5% cash back on up to $1,500 in combined purchases in bonus categories each quarter you activate (then 1%).<sup>[13]</sup><sup>[11]</sup><sup>[12]</sup>

 * For April-June 2025, the categories are Amazon and select streaming services.<sup>[12]</sup>

 * *   5% back on travel booked through Chase Travel.<sup>[15]</sup><sup>[8]</sup><sup>[16]</sup><sup>[6]</sup><sup>[12]</sup><sup>[11]</sup>

 * *   3% back on dining (including takeout and delivery) and drugstore purchases.<sup>[12]</sup><sup>[13]</sup>

 * *   1% back on all other purchases.<sup>[8]</sup><sup>[9]</sup><sup>[10]</sup><sup>[12]</sup><sup>[11]</sup>

 * *   **Other Benefits:** Includes cellphone protection, purchase protection, extended warranty, and trip cancellation/interruption insurance.<sup>[13]</sup>

 * U.S. Bank Altitude Go Visa Signature Card**<sup>[9]</sup><sup>[17]</sup><sup>[18]</sup><sup>[19]</sup><sup>[7]</sup>

 * *   **Dining Rewards:** 4x points on dining, takeout, and restaurant delivery (up to $2,000 spent per quarter, starting April 14, 2025).<sup>[9]</sup><sup>[19]</sup><sup>[18]</sup>

 * *   2x points at grocery stores, gas stations/EV charging stations, and streaming services.<sup>[9]</sup><sup>[19]</sup><sup>[18]</sup><sup>[17]</sup><sup>[7]</sup>

 * *   1x point on all other purchases.<sup>[8]</sup><sup>[9]</sup><sup>[10]</sup><sup>[18]</sup><sup>[11]</sup><sup>[12]</sup>

 * *   **Welcome Offer:** Earn 20,000 bonus points after spending $1,000 in the first 90 days.<sup>[17]</sup>

 * *   **Additional Perks:** $15 annual credit for eligible streaming services.<sup>[9]</sup>

 * *   **Important Change (Starting April 14, 2025):** Any new points will expire if there is no reward, purchase, or balance activity on your account for 12 consecutive statement cycles.<sup>[9]</sup><sup>[19]</sup><sup>[7]</sup>

 * Chase Ink Business Cards**<sup>[20]</sup><sup>[8]</sup>

 * *   **Bonus points:** Earn 90,000 bonus points after you spend $8,000 on purchases in the first 3 months from account opening.<sup>[21]</sup><sup>[22]</sup><sup>[8]</sup><sup>[11]</sup>

 * *   **Earning Rate:** Earn 3 points per $1 on the first $150,000 spent in combined purchases on travel, shipping purchases, Internet, cable and phone services, advertising purchases made with social media sites and search engines each account anniversary year.<sup>[10]</sup><sup>[11]</sup><sup>[8]</sup>

 * Earn 1 point per $1 on all other purchases.<sup>[8]</sup><sup>[11]</sup><sup>[10]</sup><sup>[12]</sup><sup>[9]</sup>

 * *   **Welcome Offer:** Earn $750 bonus cash back after you spend $6,000 on purchases in the first 3 months from account opening.<sup>[21]</sup><sup>[23]</sup><sup>[8]</sup>

 * *   **Earning Rate:** Unlimited 1.5% cash back on every purchase.<sup>[10]</sup>

 * *   **Welcome Offer:** Earn $350 when you spend $3,000 on purchases in the first three months and an additional $400 when you spend $6,000 on purchases in the first six months after account opening.<sup>[21]</sup><sup>[24]</sup><sup>[8]</sup><sup>[25]</sup>

 * *   **Earning Rate:** 5% cash back on the first $25,000 spent in combined purchases at office supply stores and on internet, cable, and phone services each account anniversary year; 2% cash back on the first $25,000 spent in combined purchases at gas stations and restaurants each account anniversary year; and 1% cash back on all other purchases.<sup>[26]</sup><sup>[10]</sup>

 * *   **For Dining:** The **U.S. Bank Altitude Go** is strong for dining, but note the quarterly cap starting April 14, 2025.<sup>[19]</sup><sup>[18]</sup>

 * *   **For Rotating Categories:** The **Chase Freedom Flex** is excellent if you can maximize the rotating quarterly bonus categories.<sup>[13]</sup>

Citations:

1. [substack.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAIO9_ZuZI5cHGGpqZyyHgTFsQdu64-45nQksWopklJwGhGfqwuPzqg3-G4GLabhEoChbrJ4h8zapPEkzmCHm3N93XtfsE4eH_uLC_eA6E1gLta5ptHfPn0iRjL7NPv6-rkY7DhdX0NGN1ln69MAE16G619uo0WcsjeM9TkP)
2. [bankrate.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqALXnj-dFuXaKHYhSX-zivUjlYtJf_h_IrX2XEEHcI5tg3ykz7efbb14_97ctP0BgKvhQTvcNcNPzl3uuwdHKO2m_Q6qN4WX1lADO6KgoqZug16j5AdtO2BgFMvZZt-OfPL60t65iPquv3vuizqiYFrxSnN8XQ==)
3. [onemileatatime.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAIPCukzMA48XLE1Wg4TQDQGkTC_-MmgH055iFQik2c3ovcaKXtKDlNlkhK4vI6iughfIkhHDsJ9UsTXTkmns3nPPZiWLvDK5PlSKlyuj7CWwWedvV4fTfZAUxV-OOndGG8Y4REHsr52hkmIpP7a23W9Oyo5RLc=)
4. [awardwallet.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJ0Wc160LWAiPCwIQa-7AWZ65HVZeBa1hXeY24l21IDyvJz-YMEeos7NNTi5WoyIsqPYyd0E7AyqMUEsVo7XWzLtCUEw1HIWnOMUWyAmRkrL4pfl0z0b4P12y_S_8XAx6tVHwxjT4azqcwnAv9hm200uw==)
5. [thebulkheadseat.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAL5s_24s9DZlTG8XxT_rivSGj_hM4jQ5aoqjiGNx4H85RfcypUIBKqeWCuF_iDBXQfzYr2B99WKR_ug5Lu0g5D9Qm_G99m4agFRGNuMGzup0szLPvoD6Cpl29pgegnXGdO8gOw8QCradSEPeEd4vGLQoTKLiz81LCX4V1AQ6OTl0Byc5LcMiboVPBEoQnmsaZUAQ7adcQngIHHE6dGJa0pIWSet5j7gpGeF-yHbxeYpCVA9YQ==)
6. [travelfreely.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAICJ-3eVuf0oGQlfXO9Ecq5KlgbrOSOXaWVrVD91CCsNcGYNDksdurIVkd6Ou8SS6wHb5CLrbLQf1_o7ApzSGpFioTQl4RAQjbxGemKTHCyxyhPqvIrNhTgb1LMGFJ-Cs-I81bOYoOAXe8hHx--rx9MBABh29qqzZgDRmtHFIKh2A==)
7. [reddit.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqALPCe6k2wCTJHXeibf2fZRf-gsOwxmMK2UwZKX0E3z3mVCeuhDao5HbOfIcUeZTlH8c6YlIRsAQuyFVV0U--WM9mf3TAcQ5OTBsnyIrJQWjUxnDdyQMelP2N2KEJjCkjhUD1pZbYCwPMHqyfBCzS8MLaBUzCR1yYcR_wbHwzLKdgoooymtW7GpVC2tf8XYv1eBdl3hZFSJxKLyQKITX)
8. [awardwallet.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJvjrM0Rz4hRGT_y1lW9_lgoB4Co4g0TjIq2tZlE_UWF222MvQP2JI0cOdKb-CjQe4BADoqLyNdCGPuDNZDAyvnjgDa55BMJ22KXzyEmYXBjf4jHPuSQF_baH7yCzZy05086kBBhj7QVdAkoN4wrYV0jeHUktPY6WCO)
9. [usbank.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJh_Y3GryNGx9m5xy3rDJZQvo-k4FPyHYqCNCHsM8G0qijOiUW_ZCGy5XucbYPrApM_fm7lvlJP4Q05isOxtxLvWsRwhEmMZBWizCbYy-4ZARafPguA3juP_iXCWA3XXkagXnOyTxtV3Q2Mn519TdyMjMrdEML1SfFqTaD7wk_aGQMyrOh8jurgZVU=)
10. [financebuzz.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAKXmioVlk5ew51-bC7wHFpw_1jK99M8Wz4qliElTPrG-nrGjE0eYlOEbExkIaRs0U4zsg5CTWajGwuTHCy3LmBqyzU3FIeXAspOQdEkWnWnc7SaFDp2c5SBoYtl_es403Ky80FpaCR4lhj8wx0ZJNLwzHL43x0z)
11. [lendingtree.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJrP3jiYFB_XxJfTdQljen7LbWe1MJpvrwK5Tkn98nMKz2AuLJh5I6AvtHwg3mVKtn9ToKLOhFGVGZFRCuhcDIrhWT05mSwJWWreq1TkyaAiDDHRi5uD8EyA0ilAbcWifgcQkba9WPw_qLB12XoE1L4DyXL4jXUBEs=)
12. [nerdwallet.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqALfAU_K-mnCqGoZfKg1253ecTROlaERD12F5FatTczC2bIh7hyuFLuNuMhZDxgiOQR3YdGjXJF_h2E1wZ5I-9ACvezZhTkIRKja5_6xXKwA9rf1eEpdE_UfDwmmredfdAa73sQyNhxGc1lwy44qehyi6Xv2AXV6uXBHvGOjJrrFgWyZOywnEEmmAsw0iIqWAD0w-15djUnwBGHGdvk5lF8WPw==)
13. [erika.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJ5ZaY4YtKjf2hyMzqw7Ga6xync48I0akWRjjQRnKdBHqwvClBa1FtOfgaYCzefNKi02QbMi3GiBM69tlUSV9gMfsoHsjwFMRmWKwDASu_cnjqFmOX5Papv7T4jmh2belg-fj4QAg==)
14. [nerdwallet.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJQfyxJb24BnvSOKwy4UfVzLl_k2AyoX3uwOj9dKzJdr45wQuzvNICPwSOYqUVlp82CCaIe9H7nLnx8dw8AlRu7Y9aZxHolGm_ZjK6l0S97B0nVxrJ2JYPAuTJYAf-K4EgWwIYFCdMB1FmBvUkSuv0=)
15. [10xtravel.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAIllcgMjAwZRXfD1y6NcnXjQJvRl7AXWL9nXZOkXjSet7Gpb-Sp3Cqdy_FT4HZUgJOJEM_HrXPG1_cpRjg-x-vRQZvF63X8dR0fLQ_Bm6Yun-MOiDl7gr2mc0D7DH7iuLtuGzam-o7z)
16. [creditkarma.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAIeLadOP51Nl2laZLlgOVeiIyj2CSnadpOONRSoxIGQSPPaP2JQZHEg9HStuatVxAvRfPHnk8ZklIFYLakuRWgvzcdprzGyNAEmjCB7Z3zBppgRTw70EipvCgs_7G61IoS1lH9Dt-6rA6if0i2XluQ8m5_URrGm5LXDZGwGCR03sTXeg38Du30uNozBbC1lCPGA7khT53Z60E4soFODH27cQ0U8sW2ghgC0lB3nTCFn2Uj_Pg3Z5vz3tq5Dr0KVG4_XqPCy328kmhvCl2cg-sBQl4lUHgCrGpNj5G6qcpln6Ben7iKp4bRHb3HL)
17. [forbes.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAL5wChb2Jsa4ICVSJRmD-J7oiR_AZHW9B-UqWRYOZmOLjn7p5L79pmNWdkWOHA9ESGGhLh60L5Ikwgi67V2D81cgElRQ0gwBgBzFpSJZXJhCn22uPcPaMxfxLa-dl9_armxgIMKP5cQ9pFBWei0uFQ8vpjgeZMr-iuDYtmMnJiIPU97OSmDy4igfTR9XwudrsBP1gg=)
18. [moneygeek.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAIriZrHGGmKCkAGrmEgwVD5lNn5C4-qIHnkNsUybN07fxKRgJuiruvezk3ZN2cG17TsMRrNPu9zBZpsX7Tyn48fo289mBjDrjN7E3nI5Cig50jOmWhyVuotrZGu2vUblFHGiAgHkzhziJrFHZE-TaBPv2Ig2ywEtfwG4JbZKdjCPlgpYaFYOXAW-bwGZtNfhLz-FEqnVA==)
19. [upgradedpoints.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJb_Oho-L7MIJtVrOztMWvbIbQHP1Xkf9zHpvpb7Sqxw-e_o2eIVyl3cdz0z3rpZTOsKiSttLrc9UijD2SiTJxTRdx3cIqlfELM4BplJPfuQMA6rVTnMOxoRUSQojQi-EibB_pAMcv-17q-4LKpFp-VljTMTvvpFA==)
20. [frequentmiler.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAIhPzsPq1wpdBmfsdWh6r8VTJ2U_pCiM3dQGSr6G-xM-yuec2lcrTQsKzwAUqafdqVo_VEGiUR0cMq4J0irB3TnwWMr3qyfR1rkSml7S2GvDDN96Xr4gmxLe58=)
21. [thepointsguy.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAIsBdk-tddpZ-c14B1ZL1nG6jC7hxcR1pbK6s3U7KP3k6S_lcGrgw01ZSu1KKJgfaCtVzXVnso9p0aKh52ZXRYmf8GfqqyBsE8g31ycZa6qS25v6E430KTqbMLy4P98xz1vi-5nuUybcZWZd7kXdiA7vvFIrCxhDD0QWCU=)
22. [dailydrop.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAKen-dJSk8IcoyMy9l3ApxmjbcaDQ7j4hZmxCYixVfCBATISeeauRNN-sfDKUw1eN4KEo9T-pfsy_QwtDhaVl743I6Yaft8HTsjz_-0KgXWOrhl1G1uVIGMiLGSV1eKUx3usJGHnEsuVra0siERt8cPFsUuMU4OtidNhe_thVvp-7CZsnk=)
23. [dailydrop.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAKgLlGhq8mDwZUzyGWA-NFos1jjhaR9000QaAutU8PKK9aRZ4HW9xpEDKrj_ztWwvPN-H_-eDP8X6JoQV_0MKfQHYyl2QSY53PojvoBoAWqgfIaueRET2NOSaSdLLAchWrAI0o_kPWN1Nsa5a354rPKofhQs7OMBy4MnvybcLrKo6vPhug=)
24. [chase.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJFZukSp4rCRhd0Iuw4AR1ZV9OZ3OiQ3p74AlTD7z22liyU5aasVUjwlEy_fs0aGkhJS9mftZLQ1PmDVkowDpHtPUx96SEHZlSd90q62HDIaVmt8i9Y6SQhjHhUtojKZ51xXg4UyhtCPN8R5YfoLIvl7YTkQyhpXQ==)
25. [travelmomsquad.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqALuAdoN1w0p_mVHpsgg0jMZDj888eWtwOAbHZVxyib1pzu8WGNAJj1xA_kKxCD81YTlef_u6E8sCjR899AYibumb0-RMohR0bP6LDs4GVJjJXNfvEYy9X2BC5CGHVMXkBMylZuPUpVeMblklJwVNe4uBE4jpe3em-Wbr9E67FErafw=)
26. [dailydrop.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJ6Tfd8NOk28dg4IVpveNPdFTboIqQmVWM13gZ2Fzersf4xrwxVMTzy6dDwFhQnnS9Bx9KOaMqyzF0kHTgCwEA-Z7llept3I1zG47jrykLCltxp3xZ-czEDVLp-28NQ0B9JzOWFlEdLtV4tAlzrLi_rK-qvNjC8ZS8RtcDT-hqc)


# End Notes/Evaluation
To evaluate the model, we employed human feedback and a fine-tuning process that included a temperature parameter.

### Gen AI Intensive Course Capstone 2025Q1

Capstone requirements: https://www.kaggle.com/competitions/gen-ai-intensive-course-capstone-2025q1

### Authors:
- Kishan Patel: [Kaggle-kishanpatelai](https://www.kaggle.com/kishanpatelai) | [LinkedIn](https://www.linkedin.com/in/kishan-patel-dev/)

### Citation:

@misc{gen-ai-intensive-course-capstone-2025q1,
    author = {Addison Howard and Brenda Flynn and Kinjal Parekh and Myles O'Neill and Nate and Polong Lin},
    title = {Gen AI Intensive Course Capstone 2025Q1},
    year = {2025},
    howpublished = {\url{https://kaggle.com/competitions/gen-ai-intensive-course-capstone-2025q1}},
    note = {Kaggle}
}